<a href="https://colab.research.google.com/github/Chris-p05/NPL-Project/blob/main/OptiHireDataScraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch tensorflow
!pip install transformers
!pip install nltk spacy

In [ ]:
# Install the Kaggle API
!pip install kaggle
# Download the 200k dataset directly
!kaggle datasets download -d rhythmghai/resume-screening-dataset-200k-candidates
!unzip resume-screening-dataset-200k-candidates.zip

In [9]:
import scrapingbee
from scrapingbee import ScrapingBeeClient
import json

# Replace 'YOUR_API_KEY' with your actual ScrapingBee API key
client = ScrapingBeeClient(api_key='YOUR_API_KEY')
# Use AI to extract structured data from a job URL
response = client.get(
    'https://www.simplyhired.com/search?q=python+developer',
    params={
        'ai_query': 'Extract job title, company, requirements, and responsibilities',
        'render_js': True
    }
)
job_data = response.json()

In [10]:
import requests

# Optimization and matching in one call
payload = {
    # Replace 'base64_encoded_pdf_or_text' with your actual base64 encoded resume content
    "resume": "base64_encoded_pdf_or_text",
    # Replace 'Senior AI Engineer requirements...' with your actual job description text
    "job_description": "Senior AI Engineer requirements...",
    "auto_optimize": True,
    "anonymize": True # This handles PII removal automatically!
}
response = requests.post("https://resumeoptimizerpro.com/api/optimize", json=payload)

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

def redact_pii(text):
    doc = nlp(text)
    for ent in doc.ents:
        # Redact Names, Emails, and Locations for objectivity
        if ent.label_ in ["PERSON", "GPE", "DATE"]:
            text = text.replace(ent.text, "[REDACTED]")
    return text

In [ ]:
!pip install pymupdf
import pymupdf

def extract_clean_text(pdf_path):
    doc = pymupdf.open(pdf_path)
    full_text = []
    for page in doc:
        # "blocks" gives us (x0, y0, x1, y1, text, block_no, block_type)
        blocks = page.get_text("blocks")
        # Sort by x coordinate (columns) then y coordinate (top to bottom)
        blocks.sort(key=lambda b: (b[0], b[1]))
        for b in blocks:
            if b[4].strip(): # Only take blocks with text
                full_text.append(b[4])
    return "\n".join(full_text)

In [ ]:
import spacy

# Download the spaCy model if it's not already installed
try:
    nlp = spacy.load("en_core_web_md") # Medium model for better accuracy
except OSError:
    print("Downloading en_core_web_md model...")
    spacy.cli.download("en_core_web_md")
    nlp = spacy.load("en_core_web_md")

def normalize_resume(text):
    doc = nlp(text.lower())
    # Filter out stopwords and punctuation, then lemmatize
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    return " ".join(tokens)

In [ ]:
def anonymize_text(text):
    doc = nlp(text)
    redacted_text = text
    for ent in doc.ents:
        # Target specific bias-inducing labels
        if ent.label_ in ["PERSON", "GPE", "DATE"]:
            redacted_text = redacted_text.replace(ent.text, f"[{ent.label_}_REDACTED]")
    return redacted_text